# Week 5 - Apache Spark Fundamentals
### Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

In [ ]:
df = spark.table("workspace.default.sales")

### Dataset

A synthetic retail sales dataset containing 1000 records was generated using Python (Pandas + Faker) and imported into Databricks as a Spark table.

In [ ]:
df.show(5)

+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|transaction_id|user_id|transaction_date|      raw_timestamp|store_id|product_category|sale_amount| price|quantity|region|     city|age|subscription|   status|               email|      username|payment_method|
+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|           522|   1076|      2025-02-26|2025-02-26 05:35:17|       9|     Electronics|     1728.8|762.98|       9| South|  Chennai| 56|    Standard|Completed|patriciaadams@exa...|victorgonzalez|           UPI|
|           738|   1064|      2025-09-17|2025-09-17 12:23:25|      10|          Sports|    2063.39|186.14|       9| South|    Delhi| 27|    Standard|Cancell

In [ ]:
df.printSchema()

root
 |-- transaction_id: long (nullable = true)
 |-- user_id: long (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- store_id: long (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- payment_method: string (nullable = true)



#### Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

- The traditional MapReduce stores intermediate data on disk after every step, so it is slower.
- It takes more time for iterative tasks like machine learning.
- Writing MapReduce programs is more complex.
- It is not suitable for real-time or interactive data processing.
- Spark keeps data in memory, so it processes data much faster.

#### Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

- Spark stores data in RAM instead of writing to disk after every operation.
- The same data can be reused many times without reading it again.
- This reduces disk I/O and improves performance.
- It is very useful for ML because the same dataset is processed repeatedly.
- So Spark is much faster than MapReduce for iterative tasks.

#### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

In [ ]:
df_no_dup= df.dropDuplicates(["user_id", "transaction_date"])
df_no_dup.show(5)

+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|transaction_id|user_id|transaction_date|      raw_timestamp|store_id|product_category|sale_amount| price|quantity|region|     city|age|subscription|   status|               email|      username|payment_method|
+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|           522|   1076|      2025-02-26|2025-02-26 05:35:17|       9|     Electronics|     1728.8|762.98|       9| South|  Chennai| 56|    Standard|Completed|patriciaadams@exa...|victorgonzalez|           UPI|
|           738|   1064|      2025-09-17|2025-09-17 12:23:25|      10|          Sports|    2063.39|186.14|       9| South|    Delhi| 27|    Standard|Cancell

Duplicate records are removed by checking user_id and transaction_date to unique transaction for each user on a particular date.

#### Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount. 

In [ ]:
from pyspark.sql.functions import avg
df_sales = df.filter(df.region == "West")\
        .groupBy("product_category")\
        .agg(avg("sale_amount").alias("average_sale"))
df_sales.show()

+----------------+------------------+
|product_category|      average_sale|
+----------------+------------------+
|       Furniture|2826.2611764705885|
|        Clothing| 2492.181020408163|
|         Grocery|2578.5170000000003|
|     Electronics|2195.3724324324317|
|          Sports|2727.4674509803926|
|        Grocery | 2373.383333333333|
|       Clothing |          3803.445|
|         Sports |           1318.62|
|      Furniture |           2535.84|
+----------------+------------------+



The data is first filtered for the West region and then grouped by product category. The average sale amount is calculated for each category.

#### Q5. What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

- .na.drop() removes rows that contain null values.
- .na.fill() keeps the rows and replaces null values with a given value.
- We use .na.fill() when we don't want to lose data.


In [ ]:
df_fill = df.na.fill({"status": "Unknown"})
df_fill.select("status").show(10)

+---------+
|   status|
+---------+
|Completed|
|Cancelled|
|Completed|
|  Pending|
|Cancelled|
|Completed|
|Cancelled|
|  Pending|
|Cancelled|
|Cancelled|
+---------+
only showing top 10 rows


Instead of deleting records with missing status, the null values are replaced with "Unknown" to preserve the data for further analysis.

#### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

In [ ]:
from pyspark.sql.functions import count
city_cnt = df.groupBy("city")\
            .agg(count("*").alias("total_records"))\
            .filter("total_records > 100")

city_cnt.show()

The dataset is grouped by city and the total number of records is counted.Only cities with more than 100 records are displayed.

#### Q7. How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

- Spark DataFrames are immutable, which means they cannot be changed directly.
- Every operation creates a new DataFrame.
- The original DataFrame remains unchanged.
- This helps avoid accidental data loss.
- So we store the result in a new DataFrame or overwrite the existing one.

#### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

In [ ]:
df_filter_age = df.filter(
 (df.age.between(18, 30)) &   (df.subscription =="Premium")
)
df_filter_age.show()

+--------------+-------+----------------+-------------------+--------+----------------+-----------+-------+--------+------+---------+---+------------+---------+--------------------+----------------+--------------+
|transaction_id|user_id|transaction_date|      raw_timestamp|store_id|product_category|sale_amount|  price|quantity|region|     city|age|subscription|   status|               email|        username|payment_method|
+--------------+-------+----------------+-------------------+--------+----------------+-----------+-------+--------+------+---------+---+------------+---------+--------------------+----------------+--------------+
|           679|   1253|      2025-07-02|2025-07-02 16:08:01|       2|       Furniture|    1814.77|1423.88|       8|  West|  Chennai| 25|     Premium|Completed|noahthornton@exam...|        dustin60|          Card|
|           637|   1142|      2026-03-11|2026-03-11 06:55:44|       5|       Furniture|     584.78| 516.37|       8| South|   Mumbai| 26|     Pr

The query filters customers who are between 18 and 30 years old and have a Premium subscription.

#### Q9. When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

- Null values can affect the accuracy of calculations.
- Some records may be ignored during aggregation.
- Filling or removing null values gives more meaningful results.
- Clean data improves the quality of analysis.

#### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 

In [ ]:
from pyspark.sql.functions import col, try_to_timestamp
from pyspark.sql.types import TimestampType
df_ts = df.withColumn(
"event_time",try_to_timestamp(col("raw_timestamp"))
).drop("raw_timestamp")
df_ts.show(5)

+--------------+-------+----------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+-------------------+
|transaction_id|user_id|transaction_date|store_id|product_category|sale_amount| price|quantity|region|     city|age|subscription|   status|               email|      username|payment_method|         event_time|
+--------------+-------+----------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+-------------------+
|           522|   1076|      2025-02-26|       9|     Electronics|     1728.8|762.98|       9| South|  Chennai| 56|    Standard|Completed|patriciaadams@exa...|victorgonzalez|           UPI|2025-02-26 05:35:17|
|           738|   1064|      2025-09-17|      10|          Sports|    2063.39|186.14|       9| South|    Delhi| 27|    Standard|Cancelled|  lgreen@example.

The raw_timestamp column is converted into TimestampType and stored as event_time to perform date and time operations later.

#### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 

- Shuffle happens when Spark moves data between partitions.
- During groupBy() records with the same key are brought together.
- This data movement is called Shuffle.
- It is called a wide transformation because data is exchanged between partitions.
- Shuffle takes more time than narrow transformations.

In [ ]:
from pyspark.sql.functions import sum
revenue = df.groupBy("store_id")\
        .agg(sum("sale_amount").alias("total_revenue"))
revenue.show()

+--------+------------------+
|store_id|     total_revenue|
+--------+------------------+
|       9|109974.43999999997|
|      10|         140080.79|
|       2|140958.82000000004|
|      18|135611.75999999995|
|      11| 97801.43999999999|
|      13|114874.70000000001|
|       6|135585.01999999996|
|       5|         113061.07|
|       3|116158.98999999999|
|       7|128447.93999999996|
|      14|         115406.94|
|      15|         160015.69|
|      19|         101087.85|
|       4|148029.43999999997|
|      20|143543.40000000002|
|      12|         104792.48|
|       1| 81336.90000000001|
|       8|135270.75000000006|
|      16|162924.83000000002|
|      17| 88603.09000000001|
+--------+------------------+



The groupBy() operation groups records having the same store_id. Spark performs a shuffle before calculating the total revenue.

#### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string. 

In [ ]:
from pyspark.sql.functions import col
df_cleaned = df.filter(
    col("email").isNotNull() &   (col("username")!= "")
)
df_cleaned.show(5)

+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|transaction_id|user_id|transaction_date|      raw_timestamp|store_id|product_category|sale_amount| price|quantity|region|     city|age|subscription|   status|               email|      username|payment_method|
+--------------+-------+----------------+-------------------+--------+----------------+-----------+------+--------+------+---------+---+------------+---------+--------------------+--------------+--------------+
|           522|   1076|      2025-02-26|2025-02-26 05:35:17|       9|     Electronics|     1728.8|762.98|       9| South|  Chennai| 56|    Standard|Completed|patriciaadams@exa...|victorgonzalez|           UPI|
|           738|   1064|      2025-09-17|2025-09-17 12:23:25|      10|          Sports|    2063.39|186.14|       9| South|    Delhi| 27|    Standard|Cancell

Rows having missing email or empty username are removed to improve data quality.

#### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?  

In [ ]:
from pyspark.sql.functions import min, max, avg
df.select(
min("price").alias("Minimum Price"),
max("price").alias("Maximum Price"),
avg("price").alias("Average Price")
).show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|        22.89|      1999.91|992.7898666666673|
+-------------+-------------+-----------------+



Multiple aggregate functions are calculated together using a single query.

#### Q14. In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

- Spark may detect the wrong data type.
- Invalid dates may become NULL or cause errors.
- Some columns may be read as String instead of Timestamp.
- It can lead to incorrect analysis if the data is not cleaned first.
- It is better to check and clean the data before using it.

#### Q15: Write a final processing pipeline that: 

1.Filters out duplicates. 

2.Fills null prices with 0. 

3.Groups by store_id to calculate total revenue. 

In [ ]:
from pyspark.sql.functions import sum,round
final_df = df.dropDuplicates()\
    .na.fill({"price": 0})\
    .groupBy("store_id")\
    .agg(round(sum("sale_amount"), 2).alias("total_revenue"))
final_df.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       9|    107663.52|
|      10|    135148.57|
|       2|    140958.82|
|      18|     129113.2|
|      11|     97801.44|
|      13|     114874.7|
|       6|    135199.34|
|       5|    113061.07|
|       3|     113365.9|
|       7|    128447.94|
|      14|    111279.57|
|      15|    155971.43|
|      19|     93606.69|
|       4|    144921.68|
|      20|     143543.4|
|      12|    104792.48|
|       1|     79608.95|
|       8|    135270.75|
|      16|    157990.97|
|      17|     88603.09|
+--------+-------------+



The pipeline removes duplicate records,fills missing prices with 0, groups the data by store_id, and calculates the total revenue for each store.This combines data cleaning and aggregation in one workflow.